# Lab - EDA Bivariate Analysis: Amazon UK Product Insights Part II

Este laboratorio realiza un análisis exploratorio bivariado de productos en Amazon Reino Unido para extraer insights de negocio sobre categorías, precios, ratings y best-sellers.

In [ ]:
# Librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency
import statsmodels.api as sm

# Configuración de gráficos
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10,6)

# Cargar dataset
data = pd.read_csv('ruta/del/archivo/amazon_products_uk.csv')
data.head()

## Parte 1: Analyzing Best-Seller Trends Across Product Categories

In [ ]:
# Crosstab categoría vs best-seller
crosstab = pd.crosstab(data['category'], data['isBestSeller'])
crosstab['proportion_best_seller'] = crosstab[1] / (crosstab[0] + crosstab[1])
crosstab_sorted = crosstab.sort_values('proportion_best_seller', ascending=False)
crosstab_sorted.head(10)

In [ ]:
# Chi-square test
chi2, p, dof, ex = chi2_contingency(pd.crosstab(data['category'], data['isBestSeller']))
chi2, p

### Stacked bar chart of best-seller status by category

In [ ]:
crosstab_plot = pd.crosstab(data['category'], data['isBestSeller'])
crosstab_plot.plot(kind='bar', stacked=True, colormap='viridis')
plt.title('Best-seller status by Product Category')
plt.xlabel('Category')
plt.ylabel('Count')
plt.show()

## Parte 2: Exploring Product Prices and Ratings Across Categories and Brands

In [ ]:
# Eliminar outliers en precios usando IQR
Q1 = data['price'].quantile(0.25)
Q3 = data['price'].quantile(0.75)
IQR = Q3 - Q1
data_no_outliers = data[(data['price'] >= Q1 - 1.5*IQR) & (data['price'] <= Q3 + 1.5*IQR)]
data_no_outliers.shape

### Violin Plot: Price Distribution Across Top 20 Categories

In [ ]:
top20_categories = data_no_outliers['category'].value_counts().head(20).index
sns.violinplot(x='category', y='price', data=data_no_outliers[data_no_outliers['category'].isin(top20_categories)])
plt.xticks(rotation=90)
plt.title('Price distribution across top 20 categories')
plt.show()

### Bar chart: Average price for top 10 categories

In [ ]:
top10_categories = data_no_outliers['category'].value_counts().head(10).index
avg_price = data_no_outliers.groupby('category')['price'].mean().loc[top10_categories]
avg_price.plot(kind='bar', color='skyblue')
plt.title('Average price for top 10 categories')
plt.ylabel('Average Price')
plt.xticks(rotation=45)
plt.show()

### Box Plot: Ratings across top 10 categories

In [ ]:
sns.boxplot(x='category', y='rating', data=data_no_outliers[data_no_outliers['category'].isin(top10_categories)])
plt.xticks(rotation=90)
plt.title('Product ratings across top 10 categories')
plt.show()

## Parte 3: Relationship Between Price and Ratings

In [ ]:
# Correlation coefficient
corr_price_rating = data_no_outliers['price'].corr(data_no_outliers['rating'])
corr_price_rating

In [ ]:
# Scatter plot
sns.scatterplot(x='rating', y='price', data=data_no_outliers)
plt.title('Scatter plot of Rating vs Price')
plt.show()

In [ ]:
# Heatmap of correlations
sns.heatmap(data_no_outliers.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# QQ plot para ver si precios siguen una distribución normal
sm.qqplot(data_no_outliers['price'], line='s')
plt.show()